# 05 — Silver Application

**Credit Risk Intelligence Platform** — Camada Silver

Este notebook transforma as tabelas Bronze `application_train` e `application_test` em tabelas Silver tratadas, padronizadas e preparadas para análise e Machine Learning.

## Pipeline

```
credit_risk.bronze.application_train  →  credit_risk.silver.application_train
credit_risk.bronze.application_test   →  credit_risk.silver.application_test
```

## Transformações aplicadas

1. **Tratamento de tipos** — validação e padronização
2. **Tratamento de NULL** — estratégia por tipo de variável (categórica → 'Unknown', numérica → preservar NULL)
3. **Valores especiais** — DAYS_EMPLOYED = 365243 → NULL + flag de anomalia
4. **Padronização de categorias** — trim e normalização quando necessário
5. **Validação de regras de negócio** — flags para valores inválidos
6. **Colunas de controle** — timestamp, versão, origem, hash
7. **Auditoria** — registro completo da transformação

## Regras

> A Bronze **NÃO é modificada**. Todas as transformações criam novas tabelas Silver.
> Nenhum registro é removido sem justificativa documentada.
> TARGET é preservado sem balanceamento.

In [0]:
# ============================================================================
# CÉLULA 1 — Configuração, Imports e Parâmetros
# ============================================================================
from pyspark.sql import functions as F, types as T, Window
from datetime import datetime, timezone
import uuid

# ----------------------------------------------------------------------------
# Parâmetros do pipeline
# ----------------------------------------------------------------------------
PIPELINE_VERSION = "silver_v1.0"
NOTEBOOK_NAME = "05_silver_application"
NOTEBOOK_PATH = "/Users/abraaojose.100@gmail.com/Projeto_classificação/Projeto_Credit_Risk/05_silver_application"
EXECUTION_ID = str(uuid.uuid4())
BATCH_ID = f"silver_app_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
EXECUTION_TIMESTAMP = datetime.now(timezone.utc)

# ----------------------------------------------------------------------------
# Tabelas de origem (Bronze) e destino (Silver)
# ----------------------------------------------------------------------------
BRONZE_TRAIN = "credit_risk.bronze.application_train"
BRONZE_TEST = "credit_risk.bronze.application_test"
SILVER_TRAIN = "credit_risk.silver.application_train"
SILVER_TEST = "credit_risk.silver.application_test"
AUDIT_TABLE = "credit_risk.silver.audit_transformation"

# ----------------------------------------------------------------------------
# Colunas de metadados Bronze a remover na Silver (serão substituídas por colunas de controle Silver)
# ----------------------------------------------------------------------------
BRONZE_META_COLS = ["_ingestion_timestamp", "_source_file"]

# ----------------------------------------------------------------------------
# Criar schema Silver se não existir
# ----------------------------------------------------------------------------
spark.sql("CREATE SCHEMA IF NOT EXISTS credit_risk.silver")
print(f"Schema credit_risk.silver verificado/criado.")

# ----------------------------------------------------------------------------
# Dicionário para registrar transformações aplicadas (para auditoria)
# ----------------------------------------------------------------------------
TRANSFORMATION_LOG = []

def log_transform(table_name, step, description, records_affected=0):
    """Registra uma transformação aplicada para auditoria."""
    TRANSFORMATION_LOG.append({
        "table": table_name,
        "step": step,
        "description": description,
        "records_affected": records_affected,
    })

print(f"⏱️ Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline Version: {PIPELINE_VERSION}")

In [0]:
# ============================================================================
# CÉLULA 2 — Inspeção Inicial das Tabelas Bronze
# ============================================================================
# Carrega os DataFrames Bronze (sem modificá-los)
df_train_bronze = spark.table(BRONZE_TRAIN)
df_test_bronze = spark.table(BRONZE_TEST)

# Métricas básicas
train_row_count = df_train_bronze.count()
test_row_count = df_test_bronze.count()
train_col_count = len(df_train_bronze.columns)
test_col_count = len(df_test_bronze.columns)

print("=" * 70)
print("INSPEÇÃO INICIAL — BRONZE")
print("=" * 70)
print(f"\n📊 {BRONZE_TRAIN}")
print(f"   Registros: {train_row_count:,}")
print(f"   Colunas: {train_col_count}")

print(f"\n📊 {BRONZE_TEST}")
print(f"   Registros: {test_row_count:,}")
print(f"   Colunas: {test_col_count}")

# ----------------------------------------------------------------------------
# Schema detalhado (tipos e nullable)
# ----------------------------------------------------------------------------
print(f"\n{'─' * 70}")
print("SCHEMA — application_train (tipos)")
print("─" * 70)
schema_train = df_train_bronze.schema
for field in schema_train.fields[:20]:  # Primeiras 20 colunas para exibição
    print(f"   {field.name:<40} {field.dataType.simpleString():<10} nullable={field.nullable}")
print(f"   ... ({train_col_count} colunas no total)")

# ----------------------------------------------------------------------------
# Duplicidades por SK_ID_CURR
# ----------------------------------------------------------------------------
train_dup_sk = train_row_count - df_train_bronze.select("SK_ID_CURR").distinct().count()
test_dup_sk = test_row_count - df_test_bronze.select("SK_ID_CURR").distinct().count()
print(f"\n{'─' * 70}")
print("DUPLICIDADES — SK_ID_CURR")
print("─" * 70)
print(f"   application_train: {train_dup_sk} duplicatas (de {train_row_count:,} registros)")
print(f"   application_test:  {test_dup_sk} duplicatas (de {test_row_count:,} registros)")

# ----------------------------------------------------------------------------
# Distribuição do TARGET (apenas application_train)
# ----------------------------------------------------------------------------
print(f"\n{'─' * 70}")
print("DISTRIBUIÇÃO DO TARGET (application_train)")
print("─" * 70)
target_dist = df_train_bronze.groupBy("TARGET").count().orderBy("TARGET")
display(target_dist)
target_rows = target_dist.collect()
for row in target_rows:
    pct = row["count"] / train_row_count * 100
    print(f"   TARGET={row['TARGET']}: {row['count']:,} ({pct:.2f}%)")

# ----------------------------------------------------------------------------
# Anomalia DAYS_EMPLOYED = 365243
# ----------------------------------------------------------------------------
days_365243 = df_train_bronze.filter(F.col("DAYS_EMPLOYED") == 365243).count()
print(f"\n{'─' * 70}")
print("ANOMALIA DAYS_EMPLOYED = 365243")
print("─" * 70)
print(f"   Registros com 365243: {days_365243:,} ({days_365243/train_row_count*100:.2f}%)")
print(f"   → Valor especial do Home Credit (aposentados/pensionistas)")
print(f"   → Tratamento: converter para NULL + criar FLAG_DAYS_EMPLOYED_ANOMALY")

In [0]:
# ============================================================================
# CÉLULA 3 — Data Quality Inicial (Bronze)
# ============================================================================
# Análise de completude (NULLs) por coluna — top 20 colunas com mais NULLs
# para cada tabela. Usado como baseline para comparar Bronze → Silver.

def compute_null_summary(df, table_name, row_count):
    """Computa resumo de NULLs para todas as colunas de um DataFrame."""
    null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]
    null_row = df.agg(*null_exprs).collect()[0]
    null_pairs = [(c, null_row[c]) for c in df.columns if null_row[c] and null_row[c] > 0]
    null_pairs.sort(key=lambda x: x[1], reverse=True)
    return null_pairs

# ----------------------------------------------------------------------------
# application_train — NULLs
# ----------------------------------------------------------------------------
train_nulls = compute_null_summary(df_train_bronze, BRONZE_TRAIN, train_row_count)
print(f"{'─' * 70}")
print(f"TOP 20 COLUNAS COM NULLs — {BRONZE_TRAIN} ({len(train_nulls)} cols com nulls)")
print(f"{'─' * 70}")
for c, n in train_nulls[:20]:
    print(f"   {c:<40} {n:>10,}  ({n/train_row_count*100:.2f}%)")

# ----------------------------------------------------------------------------
# application_test — NULLs
# ----------------------------------------------------------------------------
test_nulls = compute_null_summary(df_test_bronze, BRONZE_TEST, test_row_count)
print(f"\n{'─' * 70}")
print(f"TOP 20 COLUNAS COM NULLs — {BRONZE_TEST} ({len(test_nulls)} cols com nulls)")
print(f"{'─' * 70}")
for c, n in test_nulls[:20]:
    print(f"   {c:<40} {n:>10,}  ({n/test_row_count*100:.2f}%)")

# ----------------------------------------------------------------------------
# Valores categóricos distintos (colunas-chave)
# ----------------------------------------------------------------------------
cat_cols_inspect = [
    "NAME_CONTRACT_TYPE", "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY",
    "NAME_TYPE_SUITE", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE",
    "WEEKDAY_APPR_PROCESS_START", "FONDKAPREMONT_MODE", "HOUSETYPE_MODE",
    "WALLSMATERIAL_MODE", "EMERGENCYSTATE_MODE"
]
print(f"\n{'─' * 70}")
print("VALORES CATEGÓRICOS DISTINTOS (application_train)")
print(f"{'─' * 70}")
for c in cat_cols_inspect:
    if c in df_train_bronze.columns:
        vals = [str(r[c]) for r in df_train_bronze.select(c).distinct().limit(30).collect()]
        print(f"   {c}: {vals}")

# ----------------------------------------------------------------------------
# Valores numéricos min/max (colunas-chave)
# ----------------------------------------------------------------------------
numeric_inspect = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "CNT_CHILDREN", "DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH", "CNT_FAM_MEMBERS", "REGION_POPULATION_RELATIVE"
]
print(f"\n{'─' * 70}")
print("MIN/MAX NUMÉRICOS (application_train)")
print(f"{'─' * 70}")
for c in numeric_inspect:
    if c in df_train_bronze.columns:
        s = df_train_bronze.select(c).summary("min", "max").collect()
        print(f"   {c:<40} min={s[0][c]:>15}  max={s[1][c]:>15}")

print("\n✅ Inspeção inicial concluída!")

In [0]:
# ============================================================================
# CÉLULA 4 — Funções de Transformação Reutilizáveis
# ============================================================================
# Funções modulares aplicadas tanto em application_train quanto application_test.
# Cada função documenta a regra aplicada e retorna o DataFrame transformado.

def remove_bronze_metadata(df, table_name):
    """Remove colunas de metadados da Bronze (_ingestion_timestamp, _source_file).
    Serão substituídas por colunas de controle Silver."""
    cols_to_drop = [c for c in BRONZE_META_COLS if c in df.columns]
    if cols_to_drop:
        df = df.drop(*cols_to_drop)
        log_transform(table_name, "remove_metadata", f"Removidas colunas Bronze: {cols_to_drop}")
    return df


def treat_days_employed_anomaly(df, table_name):
    """Trata o valor especial DAYS_EMPLOYED = 365243 (aposentados/pensionistas).
    Regra: converter 365243 para NULL e criar FLAG_DAYS_EMPLOYED_ANOMALY = 1.
    Isso preserva a informação sem distorcer análises estatísticas."""
    if "DAYS_EMPLOYED" not in df.columns:
        return df

    # Conta registros afetados antes da transformação
    anomaly_count = df.filter(F.col("DAYS_EMPLOYED") == 365243).count()

    df = df.withColumn(
        "FLAG_DAYS_EMPLOYED_ANOMALY",
        F.when(F.col("DAYS_EMPLOYED") == 365243, 1).otherwise(0)
    )
    df = df.withColumn(
        "DAYS_EMPLOYED",
        F.when(F.col("DAYS_EMPLOYED") == 365243, None).otherwise(F.col("DAYS_EMPLOYED"))
    )

    log_transform(table_name, "days_employed_anomaly",
                  "DAYS_EMPLOYED=365243 → NULL + FLAG_DAYS_EMPLOYED_ANOMALY=1",
                  anomaly_count)
    print(f"   ✅ DAYS_EMPLOYED: {anomaly_count} registros tratados (365243 → NULL)")
    return df


def treat_nulls_categorical(df, table_name, row_count):
    """Trata NULLs em colunas categóricas (string) substituindo por 'Unknown'.
    Colunas numéricas mantêm NULL — imputação estatística é responsabilidade do ML prep."""
    string_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]
    total_affected = 0

    for col_name in string_cols:
        null_count = df.filter(F.col(col_name).isNull()).count()
        if null_count > 0:
            df = df.withColumn(
                col_name,
                F.when(F.col(col_name).isNull(), "Unknown").otherwise(F.col(col_name))
            )
            total_affected += null_count
            log_transform(table_name, "null_categorical",
                          f"{col_name}: NULL → 'Unknown' ({null_count} registros)",
                          null_count)

    print(f"   ✅ NULLs categóricos: {total_affected} substituições em {len([c for c in string_cols])} colunas string")
    return df


def standardize_categories(df, table_name):
    """Padroniza colunas categóricas: trim de espaços extras.
    Não altera semântica dos valores — apenas remove espaços à direita/esquerda."""
    string_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]
    for col_name in string_cols:
        df = df.withColumn(col_name, F.trim(F.col(col_name)))

    log_transform(table_name, "standardize_categories",
                  f"Trim aplicado em {len(string_cols)} colunas string")
    print(f"   ✅ Padronização: trim aplicado em {len(string_cols)} colunas string")
    return df


def add_validation_flags(df, table_name, row_count):
    """Adiciona flags de validação para valores potencialmente inválidos.
    Não remove registros — apenas marca anomalias para análise posterior."""
    flags_added = []

    # AMT_INCOME_TOTAL deve ser > 0
    if "AMT_INCOME_TOTAL" in df.columns:
        invalid = df.filter((F.col("AMT_INCOME_TOTAL") <= 0) | F.col("AMT_INCOME_TOTAL").isNull()).count()
        df = df.withColumn("FLAG_AMT_INCOME_INVALID",
            F.when((F.col("AMT_INCOME_TOTAL") <= 0) | F.col("AMT_INCOME_TOTAL").isNull(), 1).otherwise(0))
        flags_added.append(("FLAG_AMT_INCOME_INVALID", invalid))

    # AMT_CREDIT deve ser > 0
    if "AMT_CREDIT" in df.columns:
        invalid = df.filter((F.col("AMT_CREDIT") <= 0) | F.col("AMT_CREDIT").isNull()).count()
        df = df.withColumn("FLAG_AMT_CREDIT_INVALID",
            F.when((F.col("AMT_CREDIT") <= 0) | F.col("AMT_CREDIT").isNull(), 1).otherwise(0))
        flags_added.append(("FLAG_AMT_CREDIT_INVALID", invalid))

    # CNT_CHILDREN deve ser >= 0
    if "CNT_CHILDREN" in df.columns:
        invalid = df.filter(F.col("CNT_CHILDREN") < 0).count()
        df = df.withColumn("FLAG_CNT_CHILDREN_INVALID",
            F.when(F.col("CNT_CHILDREN") < 0, 1).otherwise(0))
        flags_added.append(("FLAG_CNT_CHILDREN_INVALID", invalid))

    # REGION_POPULATION_RELATIVE deve ser >= 0
    if "REGION_POPULATION_RELATIVE" in df.columns:
        invalid = df.filter(F.col("REGION_POPULATION_RELATIVE") < 0).count()
        df = df.withColumn("FLAG_REGION_POP_INVALID",
            F.when(F.col("REGION_POPULATION_RELATIVE") < 0, 1).otherwise(0))
        flags_added.append(("FLAG_REGION_POP_INVALID", invalid))

    for flag_name, invalid_count in flags_added:
        log_transform(table_name, "validation_flag",
                      f"{flag_name}: {invalid_count} registros inválidos marcados",
                      invalid_count)
        print(f"   ✅ {flag_name}: {invalid_count} registros inválidos")

    return df


def add_control_columns(df, source_table):
    """Adiciona colunas de controle técnicas da Silver."""
    now = F.current_timestamp()

    df = df.withColumn("silver_processing_timestamp", now)
    df = df.withColumn("silver_processing_date", F.current_date())
    df = df.withColumn("silver_pipeline_version", F.lit(PIPELINE_VERSION))
    df = df.withColumn("source_table", F.lit(source_table))

    # record_hash: hash MD5 de todas as colunas de dados para rastreabilidade
    data_cols = [c for c in df.columns if c not in [
        "silver_processing_timestamp", "silver_processing_date",
        "silver_pipeline_version", "source_table"
    ]]
    hash_expr = F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NULL")) for c in data_cols])
    df = df.withColumn("record_hash", F.md5(hash_expr))

    print(f"   ✅ Colunas de controle adicionadas (timestamp, date, version, source, hash)")
    return df


def apply_silver_transformations(df, table_name, source_table, row_count):
    """Aplica todas as transformações Silver em sequência."""
    print(f"\n{'─' * 60}")
    print(f"🔧 Transformando: {table_name}")
    print(f"{'─' * 60}")

    # 1. Remover metadados Bronze
    df = remove_bronze_metadata(df, table_name)

    # 2. Tratar anomalia DAYS_EMPLOYED = 365243
    df = treat_days_employed_anomaly(df, table_name)

    # 3. Padronizar categorias (trim)
    df = standardize_categories(df, table_name)

    # 4. Tratar NULLs categóricos
    df = treat_nulls_categorical(df, table_name, row_count)

    # 5. Adicionar flags de validação
    df = add_validation_flags(df, table_name, row_count)

    # 6. Adicionar colunas de controle
    df = add_control_columns(df, source_table)

    print(f"   ✅ Transformações concluídas para {table_name}")
    return df


print("✅ Funções de transformação definidas!")

In [0]:
# ============================================================================
# CÉLULA 5 — Execução das Transformações
# ============================================================================
# Aplica as transformações Silver em application_train e application_test.
# Os DataFrames Bronze originais não são modificados.

EXEC_START = datetime.now(timezone.utc)

# ----------------------------------------------------------------------------
# application_train
# ----------------------------------------------------------------------------
print("=" * 70)
print("TRANSFORMAÇÃO SILVER — application_train")
print("=" * 70)
train_start = datetime.now(timezone.utc)

df_train_silver = apply_silver_transformations(
    df_train_bronze, SILVER_TRAIN, BRONZE_TRAIN, train_row_count
)

train_end = datetime.now(timezone.utc)
train_duration = (train_end - train_start).total_seconds()
silver_train_row_count = df_train_silver.count()
silver_train_col_count = len(df_train_silver.columns)

print(f"\n   Bronze: {train_row_count:,} rows x {train_col_count} cols")
print(f"   Silver: {silver_train_row_count:,} rows x {silver_train_col_count} cols")
print(f"   Duração: {train_duration:.1f}s")

# ----------------------------------------------------------------------------
# application_test
# ----------------------------------------------------------------------------
print(f"\n{'=' * 70}")
print("TRANSFORMAÇÃO SILVER — application_test")
print("=" * 70)
test_start = datetime.now(timezone.utc)

df_test_silver = apply_silver_transformations(
    df_test_bronze, SILVER_TEST, BRONZE_TEST, test_row_count
)

test_end = datetime.now(timezone.utc)
test_duration = (test_end - test_start).total_seconds()
silver_test_row_count = df_test_silver.count()
silver_test_col_count = len(df_test_silver.columns)

print(f"\n   Bronze: {test_row_count:,} rows x {test_col_count} cols")
print(f"   Silver: {silver_test_row_count:,} rows x {silver_test_col_count} cols")
print(f"   Duração: {test_duration:.1f}s")

EXEC_END = datetime.now(timezone.utc)
TOTAL_DURATION = (EXEC_END - EXEC_START).total_seconds()

print(f"\n{'=' * 70}")
print(f"⏱️ Tempo total de transformação: {TOTAL_DURATION:.1f}s")
print(f"{'=' * 70}")

In [0]:
# ============================================================================
# CÉLULA 6 — Escrita das Tabelas Silver (Delta Lake)
# ============================================================================
# Grava as tabelas Silver usando mode("overwrite") com overwriteSchema.
# Isso é seguro porque a Silver é reconstruída a cada execução controlada.
# A Bronze NUNCA é sobrescrita.

print("=" * 70)
print("GRAVAÇÃO DAS TABELAS SILVER")
print("=" * 70)

# ----------------------------------------------------------------------------
# credit_risk.silver.application_train
# ----------------------------------------------------------------------------
print(f"\n📊 Gravando {SILVER_TRAIN}...")
write_start = datetime.now(timezone.utc)

df_train_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(SILVER_TRAIN)

write_end = datetime.now(timezone.utc)
train_write_duration = (write_end - write_start).total_seconds()
print(f"   ✅ {SILVER_TRAIN} gravada em {train_write_duration:.1f}s")
print(f"      Registros: {silver_train_row_count:,} | Colunas: {silver_train_col_count}")

# ----------------------------------------------------------------------------
# credit_risk.silver.application_test
# ----------------------------------------------------------------------------
print(f"\n📊 Gravando {SILVER_TEST}...")
write_start = datetime.now(timezone.utc)

df_test_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable(SILVER_TEST)

write_end = datetime.now(timezone.utc)
test_write_duration = (write_end - write_start).total_seconds()
print(f"   ✅ {SILVER_TEST} gravada em {test_write_duration:.1f}s")
print(f"      Registros: {silver_test_row_count:,} | Colunas: {silver_test_col_count}")

print(f"\n{'=' * 70}")
print("✅ TABELAS SILVER GRAVADAS COM SUCESSO!")
print(f"{'=' * 70}")

In [0]:
# ============================================================================
# CÉLULA 7 — Auditoria da Transformação
# ============================================================================
# Cria/atualiza a tabela credit_risk.silver.audit_transformation
# Registra metadados da execução para rastreabilidade histórica (append mode).
from pyspark.sql.types import (StructType, StructField, StringType,
    IntegerType, DoubleType, TimestampType, LongType)

# Registros de auditoria para ambas as tabelas
audit_records = [
    {
        "execution_timestamp": EXECUTION_TIMESTAMP,
        "execution_id": EXECUTION_ID,
        "batch_id": BATCH_ID,
        "source_table": BRONZE_TRAIN,
        "target_table": SILVER_TRAIN,
        "source_row_count": train_row_count,
        "target_row_count": silver_train_row_count,
        "records_inserted": silver_train_row_count,
        "records_removed": 0,
        "records_changed": silver_train_row_count,
        "processing_duration_seconds": float(train_duration + train_write_duration),
        "pipeline_version": PIPELINE_VERSION,
        "execution_status": "SUCCESS",
        "error_message": "",
    },
    {
        "execution_timestamp": EXECUTION_TIMESTAMP,
        "execution_id": EXECUTION_ID,
        "batch_id": BATCH_ID,
        "source_table": BRONZE_TEST,
        "target_table": SILVER_TEST,
        "source_row_count": test_row_count,
        "target_row_count": silver_test_row_count,
        "records_inserted": silver_test_row_count,
        "records_removed": 0,
        "records_changed": silver_test_row_count,
        "processing_duration_seconds": float(test_duration + test_write_duration),
        "pipeline_version": PIPELINE_VERSION,
        "execution_status": "SUCCESS",
        "error_message": "",
    }
]

audit_schema = StructType([
    StructField("execution_timestamp", TimestampType(), True),
    StructField("execution_id", StringType(), True),
    StructField("batch_id", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("source_row_count", LongType(), True),
    StructField("target_row_count", LongType(), True),
    StructField("records_inserted", LongType(), True),
    StructField("records_removed", IntegerType(), True),
    StructField("records_changed", LongType(), True),
    StructField("processing_duration_seconds", DoubleType(), True),
    StructField("pipeline_version", StringType(), True),
    StructField("execution_status", StringType(), True),
    StructField("error_message", StringType(), True),
])

audit_df = spark.createDataFrame(audit_records, schema=audit_schema)

print(f"📊 Persistindo auditoria em {AUDIT_TABLE}...")
audit_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable(AUDIT_TABLE)

print(f"✅ Auditoria registrada: 2 registros em {AUDIT_TABLE}")
print("\nRegistros de auditoria:")
display(spark.table(AUDIT_TABLE).orderBy(F.col("execution_timestamp").desc()).limit(10))

In [0]:
# ============================================================================
# CÉLULA 8 — Data Quality Pós-Transformação (Bronze vs Silver)
# ============================================================================
# Compara métricas de qualidade antes (Bronze) e depois (Silver) para validar
# que as transformações foram aplicadas corretamente.

def compute_dq_metrics(df, table_name):
    """Computa métricas de DQ: row_count, col_count, null_count, duplicate_count."""
    row_count = df.count()
    col_count = len(df.columns)

    # Total de NULLs (soma de todas as colunas)
    null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) for c in df.columns]
    total_nulls = df.agg(*null_exprs).collect()[0]
    null_sum = sum([total_nulls[i] for i in range(len(df.columns))])

    # Duplicatas por SK_ID_CURR
    if "SK_ID_CURR" in df.columns:
        dup_count = row_count - df.select("SK_ID_CURR").distinct().count()
    else:
        dup_count = 0

    return {
        "table": table_name,
        "row_count": row_count,
        "col_count": col_count,
        "null_count": null_sum,
        "null_percentage": round(null_sum / (row_count * col_count) * 100, 2) if row_count > 0 else 0,
        "duplicate_count": dup_count,
    }

# ----------------------------------------------------------------------------
# Ler tabelas Silver recém-criadas
# ----------------------------------------------------------------------------
df_silver_train = spark.table(SILVER_TRAIN)
df_silver_test = spark.table(SILVER_TEST)

# ----------------------------------------------------------------------------
# Métricas Bronze vs Silver — application_train
# ----------------------------------------------------------------------------
print("=" * 70)
print("DATA QUALITY: BRONZE vs SILVER")
print("=" * 70)

bronze_train_m = compute_dq_metrics(df_train_bronze, BRONZE_TRAIN)
silver_train_m = compute_dq_metrics(df_silver_train, SILVER_TRAIN)

print(f"\n📊 application_train")
print(f"{'Métrica':<30} {'Bronze':>15} {'Silver':>15} {'Delta':>15}")
print(f"{'─' * 75}")
for key in ["row_count", "col_count", "null_count", "null_percentage", "duplicate_count"]:
    b = bronze_train_m[key]
    s = silver_train_m[key]
    d = s - b
    print(f"{key:<30} {b:>15,} {s:>15,} {d:>+15,}")

# ----------------------------------------------------------------------------
# Métricas Bronze vs Silver — application_test
# ----------------------------------------------------------------------------
bronze_test_m = compute_dq_metrics(df_test_bronze, BRONZE_TEST)
silver_test_m = compute_dq_metrics(df_silver_test, SILVER_TEST)

print(f"\n📊 application_test")
print(f"{'Métrica':<30} {'Bronze':>15} {'Silver':>15} {'Delta':>15}")
print(f"{'─' * 75}")
for key in ["row_count", "col_count", "null_count", "null_percentage", "duplicate_count"]:
    b = bronze_test_m[key]
    s = silver_test_m[key]
    d = s - b
    print(f"{key:<30} {b:>15,} {s:>15,} {d:>+15,}")

# ----------------------------------------------------------------------------
# Verificações específicas
# ----------------------------------------------------------------------------
print(f"\n{'─' * 70}")
print("VERIFICAÇÕES ESPECÍFICAS")
print(f"{'─' * 70}")

# DAYS_EMPLOYED: nenhum valor 365243 deve existir na Silver
de_365243_silver = df_silver_train.filter(F.col("DAYS_EMPLOYED") == 365243).count()
print(f"   DAYS_EMPLOYED=365243 na Silver (train): {de_365243_silver} (esperado: 0)")

# FLAG_DAYS_EMPLOYED_ANOMALY
de_flag_count = df_silver_train.filter(F.col("FLAG_DAYS_EMPLOYED_ANOMALY") == 1).count()
print(f"   FLAG_DAYS_EMPLOYED_ANOMALY=1: {de_flag_count} (esperado: ~55374)")

# Colunas de controle presentes
control_cols = ["silver_processing_timestamp", "silver_processing_date",
                "silver_pipeline_version", "source_table", "record_hash"]
for c in control_cols:
    present = c in df_silver_train.columns
    print(f"   Coluna {c}: {'✅ presente' if present else '❌ ausente'}")

# Flags de validação
validation_flags = ["FLAG_AMT_INCOME_INVALID", "FLAG_AMT_CREDIT_INVALID",
                     "FLAG_CNT_CHILDREN_INVALID", "FLAG_REGION_POP_INVALID"]
print(f"\n   Flags de validação (application_train):")
for flag in validation_flags:
    if flag in df_silver_train.columns:
        count = df_silver_train.filter(F.col(flag) == 1).count()
        print(f"      {flag}: {count} registros")

# TARGET distribution preservada
print(f"\n   TARGET distribution (Silver vs Bronze):")
silver_target = df_silver_train.groupBy("TARGET").count().orderBy("TARGET").collect()
for row in silver_target:
    print(f"      TARGET={row['TARGET']}: {row['count']:,}")

print("\n✅ Data Quality pós-transformação concluída!")

In [0]:
# ============================================================================
# CÉLULA 9 — Validação Final e Amostras
# ============================================================================
# Valida que as tabelas Silver estão corretas e coerentes com a Bronze.

print("=" * 70)
print("VALIDAÇÃO FINAL — TABELAS SILVER")
print("=" * 70)

# ----------------------------------------------------------------------------
# Validação application_train
# ----------------------------------------------------------------------------
print(f"\n📊 {SILVER_TRAIN}")
print(f"{'─' * 50}")

# Comparar row count
assert silver_train_row_count == train_row_count, \
    f"Row count mismatch: Bronze={train_row_count} vs Silver={silver_train_row_count}"
print(f"   ✅ Row count: {silver_train_row_count:,} (igual à Bronze)")

# Comparar SK_ID_CURR uniqueness
silver_train_dups = silver_train_row_count - df_silver_train.select("SK_ID_CURR").distinct().count()
assert silver_train_dups == 0, f"Duplicatas encontradas: {silver_train_dups}"
print(f"   ✅ SK_ID_CURR: único (0 duplicatas)")

# Colunas Silver vs Bronze (+ colunas de controle/flags)
print(f"   Colunas Bronze: {train_col_count}")
print(f"   Colunas Silver: {silver_train_col_count}")
print(f"   Colunas adicionadas: {silver_train_col_count - train_col_count}")
print(f"     - Removidas: {len([c for c in BRONZE_META_COLS if c in df_train_bronze.columns])} (metadados Bronze)")
print(f"     - Adicionadas: FLAG_DAYS_EMPLOYED_ANOMALY + 4 flags + 5 colunas controle")

# ----------------------------------------------------------------------------
# Validação application_test
# ----------------------------------------------------------------------------
print(f"\n📊 {SILVER_TEST}")
print(f"{'─' * 50}")

assert silver_test_row_count == test_row_count, \
    f"Row count mismatch: Bronze={test_row_count} vs Silver={silver_test_row_count}"
print(f"   ✅ Row count: {silver_test_row_count:,} (igual à Bronze)")

silver_test_dups = silver_test_row_count - df_silver_test.select("SK_ID_CURR").distinct().count()
assert silver_test_dups == 0, f"Duplicatas encontradas: {silver_test_dups}"
print(f"   ✅ SK_ID_CURR: único (0 duplicatas)")

# ----------------------------------------------------------------------------
# Amostras
# ----------------------------------------------------------------------------
print(f"\n{'─' * 70}")
print(f"AMOSTRA — {SILVER_TRAIN} (primeiras 20 linhas)")
print(f"{'─' * 70}")

# Selecionar colunas-chave para exibição
sample_cols_train = [
    "SK_ID_CURR", "TARGET", "NAME_CONTRACT_TYPE", "CODE_GENDER",
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "DAYS_BIRTH", "DAYS_EMPLOYED",
    "FLAG_DAYS_EMPLOYED_ANOMALY", "silver_processing_timestamp",
    "silver_pipeline_version", "source_table", "record_hash"
]
sample_cols_train = [c for c in sample_cols_train if c in df_silver_train.columns]
display(df_silver_train.select(*sample_cols_train).limit(20))

print(f"\n{'─' * 70}")
print(f"AMOSTRA — {SILVER_TEST} (primeiras 20 linhas)")
print(f"{'─' * 70}")

sample_cols_test = [
    "SK_ID_CURR", "NAME_CONTRACT_TYPE", "CODE_GENDER",
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "DAYS_BIRTH", "DAYS_EMPLOYED",
    "FLAG_DAYS_EMPLOYED_ANOMALY", "silver_processing_timestamp",
    "silver_pipeline_version", "source_table", "record_hash"
]
sample_cols_test = [c for c in sample_cols_test if c in df_silver_test.columns]
display(df_silver_test.select(*sample_cols_test).limit(20))

print("\n✅ Validação final concluída com sucesso!")

In [0]:
# ============================================================================
# CÉLULA 10 — Resumo Final da Execução
# ============================================================================
# Exibe um resumo completo de ambas as transformações.

print("=" * 60)
print("SILVER APPLICATION - RESUMO")
print("=" * 60)

for label, src, tgt, src_rows, tgt_rows, tgt_cols, dur, trans_log in [
    ("application_train", BRONZE_TRAIN, SILVER_TRAIN, train_row_count,
     silver_train_row_count, silver_train_col_count, train_duration + train_write_duration,
     [t for t in TRANSFORMATION_LOG if t["table"] == SILVER_TRAIN]),
    ("application_test", BRONZE_TEST, SILVER_TEST, test_row_count,
     silver_test_row_count, silver_test_col_count, test_duration + test_write_duration,
     [t for t in TRANSFORMATION_LOG if t["table"] == SILVER_TEST])
]:
    print()
    print("─" * 60)
    print(f"Tabela origem:\n  {src}")
    print(f"Tabela destino:\n  {tgt}")
    print(f"\nRegistros Bronze:\n  {src_rows:,}")
    print(f"\nRegistros Silver:\n  {tgt_rows:,}")
    print(f"\nRegistros removidos:\n  0")
    print(f"\nColunas:\n  Bronze: {len(df_train_bronze.columns) if 'train' in label else len(df_test_bronze.columns)}")
    print(f"  Silver: {tgt_cols}")
    print(f"\nTransformações aplicadas ({len(trans_log)}):")
    for t in trans_log:
        print(f"  • {t['step']}: {t['description']}")
    print(f"\nStatus:\n  SUCCESS")
    print(f"\nTempo de processamento:\n  {dur:.1f} segundos")
    print("─" * 60)

print(f"\n⏱️ Tempo total: {TOTAL_DURATION:.1f}s")
print(f"📋 Execution ID: {EXECUTION_ID}")
print(f"📦 Batch ID: {BATCH_ID}")
print(f"🔧 Pipeline: {PIPELINE_VERSION}")
print(f"\n{'=' * 60}")
print("✅ PIPELINE SILVER APPLICATION CONCLUÍDO COM SUCESSO!")
print(f"{'=' * 60}")

## Transformações Aplicadas — Documentação

### 1. Remoção de metadados Bronze
Colunas `_ingestion_timestamp` e `_source_file` removidas (substituídas por colunas de controle Silver).

### 2. Anomalia DAYS_EMPLOYED = 365243
O Home Credit usa o valor `365243` para representar aposentados/pensionistas (não é um número real de dias).
- **Regra**: `DAYS_EMPLOYED = 365243` → `NULL` + `FLAG_DAYS_EMPLOYED_ANOMALY = 1`
- **Registros afetados**: ~55.374 (18%) em application_train
- **Justificativa**: Preservar a informação sem distorcer estatísticas

### 3. Padronização de categorias
- `trim()` aplicado em todas as colunas string para remover espaços extras
- Não há alteração semântica dos valores
- Valores como `'XNA'` (CODE_GENDER), `'Unknown'` (NAME_FAMILY_STATUS) e `'None'` (OCCUPATION_TYPE) são preservados — são valores originais do dataset

### 4. Tratamento de NULLs
- **Categóricos**: NULL → `'Unknown'` (preserva a informação de que o valor não foi fornecido)
- **Numéricos**: NULL preservado (imputação estatística é responsabilidade do Feature Engineering/ML)

### 5. Flags de validação
| Flag | Condição |
|------|----------|
| `FLAG_AMT_INCOME_INVALID` | `AMT_INCOME_TOTAL <= 0` ou NULL |
| `FLAG_AMT_CREDIT_INVALID` | `AMT_CREDIT <= 0` ou NULL |
| `FLAG_CNT_CHILDREN_INVALID` | `CNT_CHILDREN < 0` |
| `FLAG_REGION_POP_INVALID` | `REGION_POPULATION_RELATIVE < 0` |

### 6. Colunas de controle Silver
| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `silver_processing_timestamp` | timestamp | Momento da transformação |
| `silver_processing_date` | date | Data da transformação |
| `silver_pipeline_version` | string | Versão do pipeline (`silver_v1.0`) |
| `source_table` | string | Tabela de origem Bronze |
| `record_hash` | string | Hash MD5 de todos os campos para rastreabilidade |

### 7. TARGET
- Preservado sem balanceamento (8,07% classe minoritária)
- Sem oversampling, undersampling ou SMOTE
- Esses tratamentos pertencem à preparação do dataset de ML

### 8. Duplicidades
- `SK_ID_CURR` é único em ambas as tabelas (0 duplicatas)
- Nenhuma deduplicação foi necessária